In [ ]:
%load_ext autoreload
%autoreload 2
    
from dataclasses import dataclass
from tqdm.notebook import tqdm
from PIL import Image

import json
import glob
import numpy as np
import os
import pyrender
import shutil
import trimesh
import voxeloo

### Helper block color definitions

In [ ]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int
        
def to_flora_id(terrain_id):
    return (1 << 24) | (terrain_id & 0xffffff)

# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x323143ff),
    TerrainVoxel("bedrock", 6, 0x4d3f53ff),
    TerrainVoxel("birch_log", 12, 0xb9c8d2ff),
    TerrainVoxel("clay", 17, 0x755a56ff),
    TerrainVoxel("coal_ore", 19, 0x2f2d34ff),
    TerrainVoxel("cobblestone", 5, 0x706e76ff),
    TerrainVoxel("diamond_ore", 23, 0x36d9f1ff),
    TerrainVoxel("dirt", 2, 0x9b633cff),
    TerrainVoxel("gold_ore", 11, 0xded02cff),
    TerrainVoxel("granite", 13, 0xb2bcc7ff),
    TerrainVoxel("grass", 1, 0x30a144ff),
    TerrainVoxel("gravel", 36, 0xad8c64ff),
    TerrainVoxel("hay", 35, 0xcdb158ff),
    TerrainVoxel("limestone", 9, 0x8b7b70ff),
    TerrainVoxel("moss", 40, 0x247447ff),
    TerrainVoxel("neptunium_ore", 30, 0x394f57ff),
    TerrainVoxel("oak_log", 3, 0x774530ff),
    TerrainVoxel("pumpkin", 10, 0xc67a2bff),
    TerrainVoxel("quartzite", 7, 0x527d79ff),
    TerrainVoxel("rubber_log", 14, 0x603526ff),
    TerrainVoxel("silver_ore", 25, 0x7f9cbaff),
    TerrainVoxel("stone", 4, 0x828690ff),
    TerrainVoxel("snow", 37, 0xe5f6faff),

    # Crafted blocks
    TerrainVoxel("basalt_brick", 15, 0x323143ff),
    TerrainVoxel("basalt_carved", 41, 0x323143ff),
    TerrainVoxel("basalt_polished", 42, 0x323143ff),
    TerrainVoxel("basalt_shingles", 43, 0x323143ff),
    TerrainVoxel("birch_lumber", 16, 0xc29b44ff),
    TerrainVoxel("clay_brick", 18, 0x883b49ff),
    TerrainVoxel("clay_carved", 44, 0x883b49ff),
    TerrainVoxel("clay_polished", 45, 0x883b49ff),
    TerrainVoxel("clay_shingles", 46, 0x883b49ff),
    TerrainVoxel("cobblestone_brick", 20, 0x706e76ff),
    TerrainVoxel("cobblestone_carved", 47, 0x706e76ff),
    TerrainVoxel("cobblestone_polished", 48, 0x706e76ff),
    TerrainVoxel("cobblestone_shingles", 49, 0x706e76ff),
    TerrainVoxel("cotton_fabric", 38, 0xe9e8e9ff),
    TerrainVoxel("diamond", 21, 0xa6f6ffff),
    TerrainVoxel("gold", 24, 0xf7ca00ff),
    TerrainVoxel("granite_brick", 26, 0xb2bcc7ff),
    TerrainVoxel("granite_carved", 50, 0xb2bcc7ff),
    TerrainVoxel("granite_polished", 51, 0xb2bcc7ff),
    TerrainVoxel("granite_shingles", 52, 0xb2bcc7ff),
    TerrainVoxel("limestone_brick", 27, 0x8b7b70ff),
    TerrainVoxel("limestone_carved", 53, 0x8b7b70ff),
    TerrainVoxel("limestone_polished", 54, 0x8b7b70ff),
    TerrainVoxel("limestone_shingles", 55, 0x8b7b70ff),
    TerrainVoxel("mushroom_leather", 39, 0x7c5a45ff),
    TerrainVoxel("neptunium", 28, 0x224a3eff),
    TerrainVoxel("oak_lumber", 31, 0x8b572aff),
    TerrainVoxel("quartzite_brick", 32, 0x527d79ff),
    TerrainVoxel("quartzite_carved", 56, 0x527d79ff),
    TerrainVoxel("quartzite_polished", 57, 0x527d79ff),
    TerrainVoxel("quartzite_shingles", 58, 0x527d79ff),
    TerrainVoxel("rubber_lumber", 34, 0x764330ff),
    TerrainVoxel("silver", 33, 0xccd6dbff),
    TerrainVoxel("stone_brick", 29, 0x828690ff),
    TerrainVoxel("stone_carved", 59, 0x828690ff),
    TerrainVoxel("stone_polished", 60, 0x828690ff),
    TerrainVoxel("stone_shingles", 61, 0x828690ff),
    TerrainVoxel("template", 69, 0x990099ff),
    TerrainVoxel("wood_crate", 22, 0x8b572aff),

    # Flora
    TerrainVoxel("oak_leaf", to_flora_id(1), 0x125d37ff),
    TerrainVoxel("birch_leaf", to_flora_id(2), 0x4e6a1dff),
    TerrainVoxel("rubber_leaf", to_flora_id(3), 0x20575aff),
    TerrainVoxel("switch_grass", to_flora_id(4), 0x4bbb47ff),
    TerrainVoxel("azalea_flower", to_flora_id(5), 0xc86c8dff),
    TerrainVoxel("bell_flower", to_flora_id(6), 0x59a5d5ff),
    TerrainVoxel("dandelion_flower", to_flora_id(7), 0xd0c227ff),
    TerrainVoxel("daylily_flower", to_flora_id(8), 0xcd8216ff),
    TerrainVoxel("lilac_flower", to_flora_id(9), 0xa767f2ff),
    TerrainVoxel("rose_flower", to_flora_id(10), 0xcc374cff),
    TerrainVoxel("cotton_flower", to_flora_id(11), 0xe9e8e9ff),
    TerrainVoxel("hemp_bush", to_flora_id(12), 0x2b6230ff),
    TerrainVoxel("red_mushroom", to_flora_id(13), 0xb02f3aff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

terrain_color = np.zeros(max(tv.index for tv in terrain) + 1, dtype=np.uint32)
for tv in terrain:
    terrain_color[tv.index] = tv.color

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Load shard files

In [ ]:
OLD_DIR = "//wsl.localhost/Ubuntu/home/taylor/data/dumps/old/2022_10_24_01_06_48"
NEW_DIR = "//wsl.localhost/Ubuntu/home/taylor/data/dumps/new/2022_10_24_20_18_35"

In [ ]:
def visualize_shard(shard):
    visualize_voxels(terrain_color[shard.array()])

In [ ]:
def shard_id_from_path(path):
    return tuple(map(int, os.path.basename(path)[:-len(".diff.bin")].split("_")))

### Grab diff shards that got considerable smaller

In [ ]:
large_diffs = []
for path in tqdm(glob.glob(f"{OLD_DIR}/*.diff.bin")):
    block = voxeloo.biomes.SparseBlock_U32()
    with open(path, "rb") as f:
        block.compressed_loads(f.read())
    if len(block.values()) > 20:
        large_diffs.append([path, block]) 

In [ ]:
bad_diffs = []
for old_path, old_block in tqdm(large_diffs):
    new_path = f"{NEW_DIR}/{os.path.basename(old_path)}"
    if not os.path.exists(new_path):
        bad_diffs.append(new_path)
        
    new_block = voxeloo.biomes.SparseBlock_U32()
    with open(new_path, "rb") as f:
        new_block.compressed_loads(f.read())
    
    if len(old_block.values()) > 2 * len(new_block.values()):
        bad_diffs.append([
            shard_id_from_path(old_path),
            old_block,
            new_block,
        ])

### Visualize each of these shards

In [ ]:
for (x, y, z), old_diff, new_diff in bad_diffs:

    # Visualize seed version
    block = voxeloo.biomes.VolumeBlock_U32()
    with open(f"{NEW_DIR}/{x}_{y}_{z}.seed.bin", "rb") as f:
        block.compressed_loads(f.read())
    visualize_voxels(terrain_color[block.array()])

    # Visualize new version
    block = voxeloo.biomes.VolumeBlock_U32()
    with open(f"{NEW_DIR}/{x}_{y}_{z}.seed.bin", "rb") as f:
        block.compressed_loads(f.read())
    block.assign(new_diff)
    visualize_voxels(terrain_color[block.array()])

    # Visualize old version
    block = voxeloo.biomes.VolumeBlock_U32()
    with open(f"{NEW_DIR}/{x}_{y}_{z}.seed.bin", "rb") as f:
        block.compressed_loads(f.read())
    block.assign(old_diff)
    visualize_voxels(terrain_color[block.array()])

# Dump the recover diffs for each of the shards

In [ ]:
OUT_DIR = "//wsl.localhost/Ubuntu/home/taylor/data/recovery_diffs/"

for (x, y, z), old_diff, new_diff in bad_diffs:
    shutil.copyfile(
        f"{NEW_DIR}/{x}_{y}_{z}.meta.json",
        f"{OUT_DIR}/{x}_{y}_{z}.meta.json",
    )
    shutil.copyfile(
        f"{OLD_DIR}/{x}_{y}_{z}.diff.bin",
        f"{OUT_DIR}/{x}_{y}_{z}.diff.bin",
    )